# PM 6 Beginner Tutorial: Cleaning CityConnect Data

Welcome! This notebook will guide you through your first complete data-preparation project.

You do **not** need to know everything before starting. Read each explanation, run the example code, look at the result and then complete the small activity.

## What is the goal?

CityConnect receives municipal service requests. Management wants to know where delays happen, but the data is messy. Before analysing it, you must make it clean and trustworthy.

## Important rule

Do not manually edit the CSV or JSON files. Every correction must happen through Python code so that another person can repeat your work.


# Step 0 — Set up Python

A **library** is a collection of ready-made tools.

- `pandas` helps us work with tables.
- `NumPy` helps us work with numbers and missing values.
- `Path` helps us refer to files and folders.

Run the cell below by clicking inside it and pressing **Shift + Enter**.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

DATA_FOLDER = Path(".")
OUTPUT_FOLDER = DATA_FOLDER / "outputs"
OUTPUT_FOLDER.mkdir(exist_ok=True)

print("Setup complete.")


Setup complete.


## Checkpoint

You should see:

`Setup complete.`

If you see an error, make sure the notebook and all three raw data files are inside the same folder.


# Step 1 — Frame the problem (PA0101)

Before touching the data, explain the real-world problem.

A good problem statement answers:

1. What is going wrong?
2. Who is affected?
3. What decision must be made?
4. What information is needed?

### Example structure

> CityConnect management is concerned that some municipal service requests take too long to resolve. This may reduce resident satisfaction and make it difficult to allocate staff. The analysis must identify where delays occur and first determine whether the available data is reliable enough.

### Your activity

Write your own version below. Also identify:
- the client;
- two stakeholders;
- the decision management wants to make;
- five measurable questions.


## My problem statement

CityConnect management is concerned about delays in resolving municipal service requests submitted by residents. These delays may negatively affect resident satisfaction and reduce the efficiency of municipal services. The problem may be caused by inefficient processes, poor resource allocation or delays in specific service categories or locations. The purpose of this analysis is to clean and validate the available data, identify patterns in service request delays and provide reliable information to help management improve service delivery and allocate resources effectively.

## Stakeholders

Municipal Service Department: Responsible for receiving, assigning, managing and resolving service requests.

## Five measurable questions

1.  What is the average time taken to resolve municipal service requests?
2.  Which service categories have the longest resolution times?
3.  Which geographical areas experience the highest number of delayed requests?
4.  What percentage of service requests are resolved within the expected time frame?
5.  Which factors, such as request category, location or assigned department, are associated with longer resolution times?



# Step 2 — Load the raw data (PA0102)

To analyse data, we first load it into a **DataFrame**.

A DataFrame is like an Excel table inside Python.

We have three source files:

1. `service_requests_raw.csv`
2. `resident_profiles_raw.csv`
3. `service_centres_raw.json`

CSV files can be loaded with `pd.read_csv()`.


In [28]:

requests_raw = pd.read_csv(r"C:\Users\Sanele\Downloads\PM06 python\service_requests_raw.csv")

residents_raw = pd.read_csv(r"C:\Users\Sanele\Downloads\PM06 python\resident_profiles_raw.csv")

print("Requests loaded:", requests_raw.shape)
print("Residents loaded:", residents_raw.shape)

Requests loaded: (800, 14)
Residents loaded: (400, 8)


JSON files may have information nested inside a larger structure. We first read the JSON and then select the `centres` section.


In [29]:
import json

with open(r"C:\Users\Sanele\Downloads\PM06 python\service_centres_raw.json", "r", encoding="utf-8") as file:
    centres_json = json.load(file)

centres_raw = pd.json_normalize(centres_json["centres"])

print("Centres loaded:", centres_raw.shape)


Centres loaded: (25, 8)


## Checkpoint

You should now have three DataFrames:
- `requests_raw`
- `residents_raw`
- `centres_raw`

Use `head()` to see the first five rows.


In [30]:
display(requests_raw.head())
display(residents_raw.head())
display(centres_raw.head())


,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system
0,SR00001,R0355,2026-05-26 15:17,2026-05-27 11:00,Water,Web Portal,High,Resolved,19.73,NaN,-33.913755,31.024099,Device-003,CRM
1,SR00002,R0301,2026-02-06 20:52,NaN,Roads,Walk-in,High,In Progress,NaN,5,-32.107892,29.734713,Device-070,Mobile API
2,SR00003,R0197,2026-02-05 20:44,2026-02-08 18:51,Water,Mobile App,Medium,Closed,70.12,1,-33.350134,21.480373,Device-044,Call Centre Export
3,SR00004,R0394,2026-06-07 17:25,NaN,Waste,Walk-in,Low,In Progress,NaN,4,-29.931750,23.174613,Device-056,CRM
4,SR00005,R0349,03/02/2026 09:15,NaN,Water,Call Centre,High,In Progress,NaN,2,-27.995956,19.423921,Device-049,Call Centre Export


,resident_id,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone
0,R0001,Resident 1,53,Western Cape,Cape Metro,18211,English,Yes
1,R0002,Resident 2,21,Eastern Cape,Nelson Mandela Bay,24575,English,Yes
2,R0003,Resident 3,21,Gauteng,Ekurhuleni Metro,1500,Sesotho,No
3,R0004,Resident 4,53,Gauteng,Tshwane Metro,10852,English,Yes
4,R0005,Resident 5,53,Eastern Cape,Buffalo City,34858,isiZulu,Yes


,centre_id,centre_name,province,municipality,staff_count,monthly_capacity,operating_hours,active
0,C001,Cape Metro Service Centre 1,Western Cape,Cape Metro,23,867,08h00 to 16h30,True
1,C002,Cape Metro Service Centre 2,Western Cape,Cape Metro,46,1041,8am-4:30pm,False
2,C003,Winelands District Service Centre 3,Western Cape,Winelands District,27,432,08:00-16:30,True
3,C004,Winelands District Service Centre 4,Western Cape,Winelands District,twenty,1222,8am-4:30pm,False
4,C005,Garden Route District Service Centre 5,Western Cape,Garden Route District,45,408,08:00-16:30,True


# Step 3 — Create a source specification (PA0102)

A **source specification** explains what each dataset is and how it will be used.

Important terms:

- **Source:** where the data came from.
- **Format:** CSV, JSON, Excel or another file type.
- **Grain:** what one row represents.
- **Primary key:** a field that should uniquely identify each row.
- **Foreign key:** a field used to link one dataset to another.
- **GIGO:** “Garbage In, Garbage Out.” Poor input data can lead to poor decisions.

Complete the table below by replacing the placeholder text.


In [31]:
source_specification = pd.DataFrame([
    {
        "dataset": "service_requests_raw.csv",
        "format": "CSV",
        "one_row_represents": "One municipal service request submitted by a resident",
        "primary_key": "request_id",
        "foreign_key": "resident_id",
        "purpose": "records service_type, submission_channel, priority,	resolution_hours and satisfaction_rating",
        "possible_limitation": "Missing resolution dates, missing satisfaction ratings, inconsistent status values, invalid dates and incorrect numerical values."
    },
    {
        "dataset": "resident_profiles_raw.csv",
        "format": "CSV",
        "one_row_represents": "One resident registered on the CityConnect system",
        "primary_key": "resident_id",
        "foreign_key": "None",
        "purpose": "Stores resident information including name, age, province, municipality, monthly household income, preferred language and smartphone access.",
        "possible_limitation": "Missing resident information, invalid ages, incorrect income values, inconsistent province and municipality names, and inconsistent Yes/No values."
    },
    {
        "dataset": "service_centres_raw.json",
        "format": "JSON",
        "one_row_represents": "One municipal service centre",
        "primary_key": "centre_id",
        "foreign_key": "None",
        "purpose": "Stores service centre information including centre name, location, staff count, monthly capacity, operating hours and active status.",
        "possible_limitation": "Inconsistent operating hour formats, incorrect numerical values (e.g. staff_count recorded as 'twenty'), missing values and inconsistent active status formats."
    }
])

display(source_specification)


,dataset,format,one_row_represents,primary_key,foreign_key,purpose,possible_limitation
0,service_requests_raw.csv,CSV,One municipal service request submitted by a r...,request_id,resident_id,"records service_type, submission_channel, prio...","Missing resolution dates, missing satisfaction..."
1,resident_profiles_raw.csv,CSV,One resident registered on the CityConnect system,resident_id,None,"Stores resident information including name, ag...","Missing resident information, invalid ages, in..."
2,service_centres_raw.json,JSON,One municipal service centre,centre_id,None,Stores service centre information including ce...,"Inconsistent operating hour formats, incorrect..."


## GIGO activity

Describe three examples of how poor-quality data could cause CityConnect to make a bad decision.

1. Incorrect resident information: If resident ages, income or contact details are incorrect, CityConnect may make poor decisions when planning services and allocating resources to residents who need them.
2.  Incorrect service request data: If service requests contain duplicate records or incorrect statuses, CityConnect may report inaccurate numbers of unresolved complaints and allocate resources incorrectly.
3. Incorrect service centre information: If staff numbers or monthly service capacity are recorded incorrectly, CityConnect may assign too many requests to centres that do not have enough staff or resources to handle them.


# Step 4 — Inspect the data (PA0103)

Never start cleaning immediately. First, learn what is inside the data.

Useful commands:

- `df.shape` — number of rows and columns
- `df.head()` — first five rows
- `df.info()` — column names and data types
- `df.describe()` — numerical summary
- `df.isna().sum()` — missing values
- `df.duplicated().sum()` — duplicate rows
- `df["column"].value_counts()` — category counts


In [32]:
print("REQUEST DATA SHAPE")
print(requests_raw.shape)

print("\nMISSING VALUES")
display(requests_raw.isna().sum().sort_values(ascending=False))

print("\nDUPLICATE ROWS")
print(requests_raw.duplicated().sum())

print("\nDATA TYPES")
display(requests_raw.dtypes)


REQUEST DATA SHAPE
(800, 14)

MISSING VALUES


resolved_at            215
resolution_hours       214
satisfaction_rating    211
created_at               1
service_type             1
device_id                1
request_id               0
resident_id              0
submission_channel       0
priority                 0
status                   0
latitude                 0
longitude                0
source_system            0
dtype: int64


DUPLICATE ROWS
1

DATA TYPES


request_id              object
resident_id             object
created_at              object
resolved_at             object
service_type            object
submission_channel      object
priority                object
status                  object
resolution_hours       float64
satisfaction_rating     object
latitude               float64
longitude              float64
device_id               object
source_system           object
dtype: object

### Your activity

Repeat the same checks for:
- `residents_raw`
- `centres_raw`

Then write down at least five problems you notice.


In [33]:
# Complete the checks below.
# residents_raw checks
# Check the first 5 rows
print("RESIDENTS DATA - FIRST 5 ROWS")
display(residents_raw.head())
 
# Check column names and data types
print("\nRESIDENTS DATA - INFORMATION")
residents_raw.info()
 
# Check missing values
print("\nMISSING VALUES")
print(residents_raw.isnull().sum())
 
# Check duplicate records
print("\nDUPLICATE RECORDS")
print(residents_raw.duplicated().sum())
 
# Check statistical summary
print("\nSTATISTICAL SUMMARY")
display(residents_raw.describe(include="all"))
 
# centres_raw checks
# Check the first 5 rows
print("SERVICE CENTRES - FIRST 5 ROWS")
display(centres_raw.head())
 
# Check column names and data types
print("\nSERVICE CENTRES - INFORMATION")
centres_raw.info()
 
# Check missing values
print("\nMISSING VALUES")
print(centres_raw.isnull().sum())
 
# Check duplicate records
print("\nDUPLICATE RECORDS")
print(centres_raw.duplicated().sum())
 
# Check statistical summary
print("\nSTATISTICAL SUMMARY")
display(centres_raw.describe(include="all"))


RESIDENTS DATA - FIRST 5 ROWS


,resident_id,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone
0,R0001,Resident 1,53,Western Cape,Cape Metro,18211,English,Yes
1,R0002,Resident 2,21,Eastern Cape,Nelson Mandela Bay,24575,English,Yes
2,R0003,Resident 3,21,Gauteng,Ekurhuleni Metro,1500,Sesotho,No
3,R0004,Resident 4,53,Gauteng,Tshwane Metro,10852,English,Yes
4,R0005,Resident 5,53,Eastern Cape,Buffalo City,34858,isiZulu,Yes



RESIDENTS DATA - INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   resident_id               400 non-null    object
 1   resident_name             399 non-null    object
 2   age                       400 non-null    object
 3   province                  400 non-null    object
 4   municipality              399 non-null    object
 5   monthly_household_income  400 non-null    object
 6   preferred_language        400 non-null    object
 7   has_smartphone            400 non-null    object
dtypes: object(8)
memory usage: 25.1+ KB

MISSING VALUES
resident_id                 0
resident_name               1
age                         0
province                    0
municipality                1
monthly_household_income    0
preferred_language          0
has_smartphone              0
dtype: int64

DUPLICATE RECOR

,resident_id,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone
count,400,399,400,400,399,400,400,400
unique,399,399,70,5,12,359,6,3
top,R0033,Resident 1,28,Eastern Cape,eThekwini Metro,1500,Sesotho,No
freq,2,1,15,108,41,41,94,221


SERVICE CENTRES - FIRST 5 ROWS


,centre_id,centre_name,province,municipality,staff_count,monthly_capacity,operating_hours,active
0,C001,Cape Metro Service Centre 1,Western Cape,Cape Metro,23,867,08h00 to 16h30,True
1,C002,Cape Metro Service Centre 2,Western Cape,Cape Metro,46,1041,8am-4:30pm,False
2,C003,Winelands District Service Centre 3,Western Cape,Winelands District,27,432,08:00-16:30,True
3,C004,Winelands District Service Centre 4,Western Cape,Winelands District,twenty,1222,8am-4:30pm,False
4,C005,Garden Route District Service Centre 5,Western Cape,Garden Route District,45,408,08:00-16:30,True



SERVICE CENTRES - INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   centre_id         25 non-null     object
 1   centre_name       25 non-null     object
 2   province          25 non-null     object
 3   municipality      25 non-null     object
 4   staff_count       25 non-null     object
 5   monthly_capacity  25 non-null     int64 
 6   operating_hours   25 non-null     object
 7   active            25 non-null     bool  
dtypes: bool(1), int64(1), object(6)
memory usage: 1.5+ KB

MISSING VALUES
centre_id           0
centre_name         0
province            0
municipality        0
staff_count         0
monthly_capacity    0
operating_hours     0
active              0
dtype: int64

DUPLICATE RECORDS
1

STATISTICAL SUMMARY


,centre_id,centre_name,province,municipality,staff_count,monthly_capacity,operating_hours,active
count,25,25,25,25,25.0,25.000000,25,25
unique,24,24,4,13,17.0,NaN,3,2
top,C006,Garden Route District Service Centre 6,Western Cape,Garden Route District,27.0,NaN,8am-4:30pm,True
freq,2,2,7,3,4.0,NaN,10,17
mean,NaN,NaN,NaN,NaN,NaN,1261.040000,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,642.980785,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,-100.000000,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,785.000000,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,1222.000000,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,1845.000000,NaN,NaN


## Five problems I found

1.  Missing values in the Service Requests dataset: The resolved_at, resolution_hours and satisfaction_rating columns contain a significant number of missing values, which may affect the accuracy of the analysis.
2.  Duplicate records in the Resident Profiles dataset: The resident_id column has 399 unique IDs out of 400 records, indicating that one resident ID is duplicated.
3.  Missing information in the Resident Profiles dataset: The resident_name and municipality columns each contain one missing value, resulting in incomplete resident information.
4.  Incorrect data types in the Service Centres dataset: The staff_count column contains the text value "twenty" instead of a numerical value, which may cause problems when performing calculations.
5.Invalid numerical values and duplicates in the Service Centres dataset: The monthly_capacity column contains a negative value (-100), which is invalid. The dataset also contains one duplicate record.


# Step 5 — Protect the raw data

We keep the original DataFrames unchanged and create copies for cleaning.

This is important because:
- we can compare before and after;
- we can restart if we make a mistake;
- we preserve the original evidence.


In [34]:
requests_clean = requests_raw.copy()
residents_clean = residents_raw.copy()
centres_clean = centres_raw.copy()

print("Cleaning copies created.")


Cleaning copies created.


##### Step 6 — Fix mixed data types

Sometimes a number is stored as text, for example:
- `"five"`
- `"R 25,000"`
- `"unknown"`

`pd.to_numeric(..., errors="coerce")` converts valid numbers and changes invalid values to `NaN`.

`NaN` means a value is missing or could not be converted.


In [35]:
# Example: convert satisfaction_rating to a number
requests_clean["satisfaction_rating"] = pd.to_numeric(
    requests_clean["satisfaction_rating"],
    errors="coerce"
)

display(requests_clean["satisfaction_rating"].value_counts(dropna=False).sort_index())


satisfaction_rating
1.0    129
2.0    122
3.0     98
4.0    124
5.0    114
7.0      1
NaN    212
Name: count, dtype: int64

### Your activity

Convert these fields safely:
- `residents_clean["age"]`
- `residents_clean["monthly_household_income"]`
- `centres_clean["staff_count"]`
- `centres_clean["monthly_capacity"]`

For income, first remove `R`, commas and spaces.


In [36]:
# Hint:
# residents_clean["monthly_household_income"] = (
#     residents_clean["monthly_household_income"]
#     .astype(str)
#     .str.replace("R", "", regex=False)
#     .str.replace(",", "", regex=False)
#     .str.strip()
# )
# residents_clean["monthly_household_income"] = pd.to_numeric(
#     residents_clean["monthly_household_income"], errors="coerce"
# )

# Complete your code here.


# 1. Convert age to numeric
residents_clean["age"] = pd.to_numeric(
    residents_clean["age"],
    errors="coerce"
)

# 2. Clean monthly household income
# Remove R, commas and spaces
residents_clean["monthly_household_income"] = (
    residents_clean["monthly_household_income"]
    .astype(str)
    .str.replace("R", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

# Convert income to numeric
residents_clean["monthly_household_income"] = pd.to_numeric(
    residents_clean["monthly_household_income"],
    errors="coerce"
)

# 3. Convert staff_count to numeric
centres_clean["staff_count"] = pd.to_numeric(
    centres_clean["staff_count"],
    errors="coerce"
)

# 4. Convert monthly_capacity to numeric
centres_clean["monthly_capacity"] = pd.to_numeric(
    centres_clean["monthly_capacity"],
    errors="coerce"
)

# Display the updated data types
print("RESIDENT DATA TYPES")
print(residents_clean.dtypes)

print("\nSERVICE CENTRE DATA TYPES")
print(centres_clean.dtypes)


RESIDENT DATA TYPES
resident_id                  object
resident_name                object
age                         float64
province                     object
municipality                 object
monthly_household_income      int64
preferred_language           object
has_smartphone               object
dtype: object

SERVICE CENTRE DATA TYPES
centre_id            object
centre_name          object
province             object
municipality         object
staff_count         float64
monthly_capacity      int64
operating_hours      object
active                 bool
dtype: object


# Step 7 — Convert dates

Dates may be stored in different formats.

`pd.to_datetime()` converts text into a real date-and-time data type.

Using `errors="coerce"` changes unrecognised dates into missing values instead of crashing the notebook.


In [38]:
requests_clean["created_at"] = pd.to_datetime(
    requests_clean["created_at"],
    errors="coerce"
)

requests_clean["resolved_at"] = pd.to_datetime(
    requests_clean["resolved_at"],
    errors="coerce"
)

display(requests_clean[["created_at", "resolved_at"]].head())


,created_at,resolved_at
0,2026-05-26 15:17:00,2026-05-27 11:00:00
1,2026-02-06 20:52:00,NaT
2,2026-02-05 20:44:00,2026-02-08 18:51:00
3,2026-06-07 17:25:00,NaT
4,NaT,NaT


### Check for impossible dates

A request cannot be resolved before it was created.


In [39]:
resolved_before_created = (
    requests_clean["resolved_at"].notna()
    & requests_clean["created_at"].notna()
    & (requests_clean["resolved_at"] < requests_clean["created_at"])
)

print("Rows resolved before creation:", resolved_before_created.sum())
display(requests_clean.loc[resolved_before_created])


Rows resolved before creation: 1


,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system
11,SR00012,R0167,2026-04-07 12:52:00,2025-12-31 10:00:00,Roads,Mobile App,Medium,Closed,69.78,2.0,-29.047625,25.775118,Device-060,Call Centre Export


Explain how you would treat these rows. Do not silently delete them.

_Write your answer here._

1. Identify and flag the records: Keep the affected rows and mark them as having an invalid date sequence so they can be investigated.

2. Investigate the source data: Check the original dataset or municipal system to determine whether the dates were entered incorrectly or whether there was a data import or formatting error.

3. Correct where possible: If the correct dates can be verified from the original source, update the incorrect dates.

4. Handle unresolved errors: If the correct dates cannot be established, change the affected resolved_at values to NaT (missing datetime value). This prevents incorrect resolution times from affecting the analysis.

5. Document the changes: Record how many rows were affected, what corrections were made and why.

# Step 8 — Standardise categories

Categories should use consistent spelling.

For example:
- `electricity`
- `Electricity`
- `electrcity`

These may all mean the same thing.

First inspect the categories.


In [40]:
for column in ["service_type", "submission_channel", "priority", "status"]:
    print("\nCOLUMN:", column)
    display(requests_clean[column].value_counts(dropna=False))



COLUMN: service_type


service_type
Electricity    168
Water          165
Sanitation     163
Waste          154
Roads          148
electrcity       1
NaN              1
Name: count, dtype: int64


COLUMN: submission_channel


submission_channel
Mobile App     226
Web Portal     197
Walk-in        190
Call Centre    186
Whatsapp         1
Name: count, dtype: int64


COLUMN: priority


priority
Medium      331
Low         272
High        151
Critical     45
URGENT        1
Name: count, dtype: int64


COLUMN: status


status
Resolved       367
Closed         218
In Progress    136
Open            78
Done             1
Name: count, dtype: int64

A mapping dictionary tells pandas how to replace incorrect values.


In [41]:
service_type_map = {
    "electrcity": "Electricity",
    "electricity": "Electricity"
}

requests_clean["service_type"] = requests_clean["service_type"].replace(service_type_map)

display(requests_clean["service_type"].value_counts(dropna=False))


service_type
Electricity    169
Water          165
Sanitation     163
Waste          154
Roads          148
NaN              1
Name: count, dtype: int64

### Your activity

Create mapping dictionaries for any other inconsistent categories you discover.

Do not guess blindly. Explain why each replacement is reasonable.


In [18]:
# Complete additional category mappings here.

# Service requests

columns_to_check = [

    "service_type",

    "submission_channel",

    "priority",

    "status"

]
 
for column in columns_to_check:

    print(f"\n{column.upper()}")

    print(requests_clean[column].value_counts(dropna=False))
 
 
# Resident profiles

resident_columns = [

    "province",

    "municipality",

    "preferred_language",

    "has_smartphone"

]
 
for column in resident_columns:

    print(f"\n{column.upper()}")

    print(residents_clean[column].value_counts(dropna=False))
 
 
# Service centres

centre_columns = [

    "province",

    "municipality",

    "operating_hours",

    "active"

]
 
for column in centre_columns:

    print(f"\n{column.upper()}")

    print(centres_clean[column].value_counts(dropna=False))
 



SERVICE_TYPE
service_type
Electricity    169
Water          165
Sanitation     163
Waste          154
Roads          148
NaN              1
Name: count, dtype: int64

SUBMISSION_CHANNEL
submission_channel
Mobile App     226
Web Portal     197
Walk-in        190
Call Centre    186
Whatsapp         1
Name: count, dtype: int64

PRIORITY
priority
Medium      331
Low         272
High        151
Critical     45
URGENT        1
Name: count, dtype: int64

STATUS
status
Resolved       367
Closed         218
In Progress    136
Open            78
Done             1
Name: count, dtype: int64

PROVINCE
province
Eastern Cape     108
Gauteng          101
KwaZulu-Natal     99
Western Cape      91
WC                 1
Name: count, dtype: int64

MUNICIPALITY
municipality
eThekwini Metro            41
OR Tambo District          40
Johannesburg Metro         38
Buffalo City               35
Garden Route District      35
Nelson Mandela Bay         32
Tshwane Metro              32
Winelands District       

In [19]:
# 1. Service type mapping
service_type_map = {
    "electrcity": "Electricity",
    "electricity": "Electricity"
}
 
# 2. Submission channel mapping
submission_channel_map = {
    "Whatsapp": "WhatsApp"
}
 
# 3. Priority mapping
priority_map = {
    "URGENT": "Critical"
}
 
# 4. Status mapping
status_map = {
    "Done": "Resolved"
}
 
# 5. Province mapping
province_map = {
    "WC": "Western Cape"
}
 
# 6. Preferred language mapping
language_map = {
    "Zulu": "isiZulu"
}
 
# 7. Smartphone mapping
smartphone_map = {
    "Y": "Yes"
}
 
# 8. Operating hours mapping
operating_hours_map = {
    "8am-4:30pm": "08:00-16:30",
    "08h00 to 16h30": "08:00-16:30"
}
 
 
# APPLY MAPPINGS
 
requests_clean["service_type"] = requests_clean["service_type"].replace(service_type_map)
requests_clean["submission_channel"] = requests_clean["submission_channel"].replace(submission_channel_map)
requests_clean["priority"] = requests_clean["priority"].replace(priority_map)
requests_clean["status"] = requests_clean["status"].replace(status_map)
 
residents_clean["province"] = residents_clean["province"].replace(province_map)
residents_clean["preferred_language"] = residents_clean["preferred_language"].replace(language_map)
residents_clean["has_smartphone"] = residents_clean["has_smartphone"].replace(smartphone_map)
 
centres_clean["operating_hours"] = centres_clean["operating_hours"].replace(operating_hours_map)
 
 
print("All mapping dictionaries applied successfully.")

All mapping dictionaries applied successfully.


# Step 9 — Find missing values

Missing data does not always mean the row must be deleted.

Possible treatments:
- leave it missing;
- fill it using a justified value;
- add a flag;
- remove the row only when essential fields are missing;
- send it back to the source owner for correction.

Count the missing values again after type conversion.


In [20]:
missing_summary = (
    requests_clean.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(requests_clean) * 100
).round(2)

display(missing_summary)


,missing_count,missing_percentage
resolved_at,215,26.88
resolution_hours,214,26.75
satisfaction_rating,212,26.50
created_at,2,0.25
service_type,1,0.12
device_id,1,0.12
request_id,0,0.00
resident_id,0,0.00
submission_channel,0,0.00
priority,0,0.00


### Your activity

Choose treatments for at least three missing-value problems.

Explain:
1. Which field is affected?
2. How many rows are affected?
3. What treatment did you choose?
4. Why is that treatment appropriate?


Resolution_hours
- Field affected: resolution_hours
- Rows affected: 214
- Treatment: Leave the values missing until the correct resolution information is available.
- Reason: Filling these values with an average could distort the actual service resolution times and affect the analysis of municipal delays.

Satisfaction_rating
- Field affected: satisfaction_rating
- Rows affected: 212
- Treatment: Leave the values missing and consider them as unavailable feedback.
- Reason: Satisfaction ratings represent residents' opinions. It would be inappropriate to assign ratings to residents who did not provide feedback.

Created_at
- Field affected: created_at
- Rows affected: 2
- Treatment: Investigate the original source records and correct the dates where possible. If they cannot be recovered, flag the records.
- Reason: The creation date is essential for calculating request processing times and identifying delays.



# Step 10 — Find duplicates

There are two different duplicate checks.

### Complete duplicate row

Every value in the row is repeated.


In [42]:
complete_duplicates = requests_clean.duplicated(keep=False)

print("Rows involved in complete duplicates:", complete_duplicates.sum())
display(requests_clean.loc[complete_duplicates].sort_values("request_id"))


Rows involved in complete duplicates: 2


,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system
249,SR00250,R0129,2026-03-24 19:41:00,NaT,Electricity,Mobile App,Medium,In Progress,NaN,1.0,-28.055828,26.986792,Device-068,Call Centre Export
250,SR00250,R0129,2026-03-24 19:41:00,NaT,Electricity,Mobile App,Medium,In Progress,NaN,1.0,-28.055828,26.986792,Device-068,Call Centre Export


### Duplicate primary key

The request ID should be unique, even if the rest of the row is different.


In [22]:
duplicate_request_ids = requests_clean.duplicated(
    subset=["request_id"],
    keep=False
)

print("Rows with duplicate request IDs:", duplicate_request_ids.sum())
display(requests_clean.loc[duplicate_request_ids].sort_values("request_id"))


Rows with duplicate request IDs: 4


,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system
249,SR00250,R0129,2026-03-24 19:41:00,NaT,Electricity,Mobile App,Medium,In Progress,NaN,1.0,-28.055828,26.986792,Device-068,Call Centre Export
250,SR00250,R0129,2026-03-24 19:41:00,NaT,Electricity,Mobile App,Medium,In Progress,NaN,1.0,-28.055828,26.986792,Device-068,Call Centre Export
300,SR00301,R0151,2026-04-30 18:59:00,2026-05-02 15:50:00,Water,Call Centre,Medium,Closed,44.87,NaN,-27.502040,22.730637,Device-010,Mobile API
301,SR00301,R0398,2026-01-16 14:24:00,NaT,Waste,Walk-in,Critical,In Progress,44.00,NaN,-31.897416,24.335875,Device-026,Call Centre Export


### Your activity

Decide how to treat:
- a completely repeated record;
- two different records with the same request ID.

Explain why these situations should not automatically receive the same treatment.

Completely repeated record
- Affected records: Rows 249 and 250, with request ID SR00250.
- Treatment: Remove one of the duplicated rows using drop_duplicates().
- Reason: Both records contain exactly the same information. Keeping both would result in double-counting the same service request and could distort the analysis.

Different records with the same request ID
- Affected records: Rows 300 and 301, with request ID SR00301.
- Treatment: Do not automatically delete either record. Flag both records for investigation and compare them against the original source system.
- Reason: Although they share the same request ID, they contain different information, including different residents, service types, dates and statuses. One record may have an incorrect request ID or may represent a data-entry error.

Why should they be treated differently?
Completely duplicated records contain identical information and can usually be safely reduced to one record. However, records with the same primary key but different information may contain important details that need investigation. Automatically deleting one could result in losing valid information.


# Step 11 — Apply business validation rules

A **validation rule** describes what values are acceptable.

Examples:
- age must be between 18 and 100;
- satisfaction rating must be between 1 and 5;
- resolution hours cannot be negative;
- monthly capacity must be positive.

Create Boolean flags. `True` means the row breaks the rule.


In [23]:
requests_clean["invalid_rating"] = (
    requests_clean["satisfaction_rating"].notna()
    & ~requests_clean["satisfaction_rating"].between(1, 5)
)

requests_clean["negative_resolution_hours"] = (
    pd.to_numeric(requests_clean["resolution_hours"], errors="coerce") < 0
)

print("Invalid ratings:", requests_clean["invalid_rating"].sum())
print("Negative resolution hours:", requests_clean["negative_resolution_hours"].sum())


Invalid ratings: 1
Negative resolution hours: 1


### Your activity

Create flags for:
- invalid ages;
- negative household income;
- invalid latitude or longitude;
- non-positive centre capacity;
- resolution dates before creation dates;
- unexpected priority and status categories.


In [43]:
# Complete validation flags here.


# 1. Invalid ages (must be between 18 and 100)
residents_clean["invalid_age"] = (
    residents_clean["age"].notna()
    & ~residents_clean["age"].between(18, 100)
)

# 2. Negative household income
residents_clean["negative_household_income"] = (
    residents_clean["monthly_household_income"] < 0
)

# 3. Invalid latitude or longitude
requests_clean["invalid_coordinates"] = (
    ~requests_clean["latitude"].between(-90, 90)
    | ~requests_clean["longitude"].between(-180, 180)
)

# 4. Non-positive centre capacity
centres_clean["invalid_monthly_capacity"] = (
    centres_clean["monthly_capacity"] <= 0
)

# 5. Resolution dates before creation dates
requests_clean["resolution_before_creation"] = (
    requests_clean["resolved_at"].notna()
    & requests_clean["created_at"].notna()
    & (requests_clean["resolved_at"] < requests_clean["created_at"])
)

# 6. Unexpected priority categories
valid_priorities = ["Low", "Medium", "High", "Critical"]

requests_clean["invalid_priority"] = (
    ~requests_clean["priority"].isin(valid_priorities)
)

# 7. Unexpected status categories
valid_statuses = ["Open", "In Progress", "Closed"]

requests_clean["invalid_status"] = (
    ~requests_clean["status"].isin(valid_statuses)
)

# Display the number of invalid records
print("INVALID AGES:", residents_clean["invalid_age"].sum())
print("NEGATIVE HOUSEHOLD INCOME:", residents_clean["negative_household_income"].sum())
print("INVALID COORDINATES:", requests_clean["invalid_coordinates"].sum())
print("NON-POSITIVE CENTRE CAPACITY:", centres_clean["invalid_monthly_capacity"].sum())
print("RESOLUTION BEFORE CREATION:", requests_clean["resolution_before_creation"].sum())
print("INVALID PRIORITIES:", requests_clean["invalid_priority"].sum())
print("INVALID STATUSES:", requests_clean["invalid_status"].sum())



INVALID AGES: 1
NEGATIVE HOUSEHOLD INCOME: 1
INVALID COORDINATES: 0
NON-POSITIVE CENTRE CAPACITY: 1
RESOLUTION BEFORE CREATION: 1
INVALID PRIORITIES: 1
INVALID STATUSES: 368


# Step 12 — Find outliers

An **outlier** is a value far away from most other values.

An outlier may be:
- a data-entry error;
- a system error;
- a real but unusual event.

Therefore, do not delete it automatically.

## IQR method

The IQR method uses the middle 50% of values.


In [24]:
resolution_numeric = pd.to_numeric(
    requests_clean["resolution_hours"],
    errors="coerce"
)

q1 = resolution_numeric.quantile(0.25)
q3 = resolution_numeric.quantile(0.75)
iqr = q3 - q1

lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

requests_clean["resolution_outlier_iqr"] = (
    resolution_numeric.notna()
    & ((resolution_numeric < lower_limit) | (resolution_numeric > upper_limit))
)

print("IQR lower limit:", lower_limit)
print("IQR upper limit:", upper_limit)
print("Outliers found:", requests_clean["resolution_outlier_iqr"].sum())


IQR lower limit: -20.91875000000001
IQR upper limit: 120.75125
Outliers found: 1


### Second method: percentile rule

Flag values above the 99th percentile.


In [25]:
p99 = resolution_numeric.quantile(0.99)

requests_clean["resolution_outlier_p99"] = (
    resolution_numeric.notna()
    & (resolution_numeric > p99)
)

print("99th percentile:", p99)
display(
    requests_clean.loc[
        requests_clean["resolution_outlier_iqr"]
        | requests_clean["resolution_outlier_p99"],
        ["request_id", "resolution_hours", "priority", "status"]
    ].sort_values("resolution_hours", ascending=False)
)


99th percentile: 112.643


,request_id,resolution_hours,priority,status
38,SR00039,9999.00,Low,Resolved
656,SR00657,120.01,Low,Resolved
178,SR00179,117.72,Low,Closed
19,SR00020,116.42,Low,Resolved
72,SR00073,113.62,Low,Resolved
214,SR00215,112.83,Low,Resolved


### Your judgement

Choose at least three flagged values and explain whether each one is:
- definitely incorrect;
- probably incorrect;
- possibly valid but unusual.

Do not remove an outlier without recording the reason.

SR00039 – 9,999 resolution hours: Probably incorrect. This value is extremely high compared with the other records and exceeds the IQR upper limit of 120.75 hours. I would investigate the original timestamps and correct it only if an error is confirmed.
SR00657 – 120.01 resolution hours: Possibly valid but unusual. This value is close to the IQR upper limit but remains below it. It could represent a genuine service-delivery delay, so I would retain it.
SR00179 – 117.72 resolution hours: Possibly valid but unusual. This value is relatively high but falls within the calculated IQR limits. I would retain it unless further investigation identifies an error.

# Step 13 — Create a data-quality log

A data-quality log is an audit trail of:
- what was wrong;
- how many rows were affected;
- what you did;
- why you did it;
- what risk remains.

Add a row to this table for every major issue.


In [26]:
quality_log_columns = [
    "dataset",
    "field",
    "issue",
    "affected_rows",
    "treatment",
    "justification",
    "remaining_risk"
]

data_quality_log = pd.DataFrame(columns=quality_log_columns)

# Example entry
example_entry = {
    "dataset": "service_requests_raw.csv",
    "field": "service_type",
    "issue": "Spelling variation: electrcity",
    "affected_rows": int((requests_raw["service_type"] == "electrcity").sum()),
    "treatment": "Mapped to Electricity",
    "justification": "The value is an obvious spelling error and Electricity is an approved category.",
    "remaining_risk": "Other unknown spelling errors may still exist."
}

data_quality_log = pd.concat(
    [data_quality_log, pd.DataFrame([example_entry])],
    ignore_index=True
)

display(data_quality_log)


,dataset,field,issue,affected_rows,treatment,justification,remaining_risk
0,service_requests_raw.csv,service_type,Spelling variation: electrcity,1,Mapped to Electricity,The value is an obvious spelling error and Ele...,Other unknown spelling errors may still exist.


### Your activity

Add entries for all the important problems you treated.


In [27]:
# Add your quality-log entries here.
quality_log_columns = [
    "dataset",
    "field",
    "issue",
    "affected_rows",
    "treatment",
    "justification",
    "remaining_risk"
]
 
quality_entries = [
    {
        "dataset": "service_requests_raw.csv",
        "field": "service_type",
        "issue": "Spelling variation: electrcity",
        "affected_rows": 1,
        "treatment": "Mapped to Electricity",
        "justification": "Obvious spelling error referring to the same service category.",
        "remaining_risk": "Other unknown spelling errors may still exist."
    },
    {
        "dataset": "service_requests_raw.csv",
        "field": "submission_channel",
        "issue": "Inconsistent capitalisation: Whatsapp",
        "affected_rows": 1,
        "treatment": "Mapped to WhatsApp",
        "justification": "Both values refer to the same messaging platform.",
        "remaining_risk": "Other inconsistent channel labels may exist."
    },
    {
        "dataset": "service_requests_raw.csv",
        "field": "satisfaction_rating",
        "issue": "Invalid rating",
        "affected_rows": 1,
        "treatment": "Flagged for investigation",
        "justification": "Ratings must fall within the expected 1 to 5 range.",
        "remaining_risk": "The original rating must be verified."
    },
    {
        "dataset": "service_requests_raw.csv",
        "field": "resolved_at",
        "issue": "Missing resolution dates",
        "affected_rows": 215,
        "treatment": "Retained missing values and flagged for investigation.",
        "justification": "Some requests may still be unresolved.",
        "remaining_risk": "Some completed requests may have missing dates."
    },
    {
        "dataset": "service_requests_raw.csv",
        "field": "resolution_hours",
        "issue": "Missing resolution times",
        "affected_rows": 214,
        "treatment": "Calculated from valid timestamps where possible; otherwise retained as missing.",
        "justification": "Valid timestamps provide a reliable way to calculate resolution time.",
        "remaining_risk": "Some missing values may remain unresolved."
    },
    {
        "dataset": "service_requests_raw.csv",
        "field": "created_at and resolved_at",
        "issue": "Resolution date before creation date",
        "affected_rows": 1,
        "treatment": "Flagged for investigation.",
        "justification": "A request cannot normally be resolved before it is created.",
        "remaining_risk": "Original timestamps require verification."
    },
    {
        "dataset": "service_requests_raw.csv",
        "field": "resolution_hours",
        "issue": "Extreme outlier: 9999 hours (SR00039)",
        "affected_rows": 1,
        "treatment": "Flagged for investigation; not automatically removed.",
        "justification": "The value exceeds the IQR upper limit of approximately 120.75 hours.",
        "remaining_risk": "The value may be genuine or caused by a recording error."
    },
    {
        "dataset": "resident_profiles_raw.csv",
        "field": "province",
        "issue": "Abbreviation: WC",
        "affected_rows": 1,
        "treatment": "Mapped to Western Cape",
        "justification": "WC is a recognised abbreviation for Western Cape.",
        "remaining_risk": "Other province abbreviations may exist."
    },
    {
        "dataset": "resident_profiles_raw.csv",
        "field": "preferred_language",
        "issue": "Inconsistent language label: Zulu",
        "affected_rows": 1,
        "treatment": "Mapped to isiZulu",
        "justification": "Both labels refer to the same language.",
        "remaining_risk": "Other language naming inconsistencies may exist."
    },
    {
        "dataset": "resident_profiles_raw.csv",
        "field": "has_smartphone",
        "issue": "Inconsistent value: Y",
        "affected_rows": 1,
        "treatment": "Mapped to Yes",
        "justification": "Y is an abbreviation for Yes in a Yes/No field.",
        "remaining_risk": "Other unexpected responses may exist."
    },
    {
        "dataset": "service_centres_raw.json",
        "field": "operating_hours",
        "issue": "Three different formats for the same operating hours",
        "affected_rows": 15,
        "treatment": "Standardised to 08:00-16:30",
        "justification": "All three formats represent the same opening and closing times.",
        "remaining_risk": "Other operating-hour variations may exist."
    }
]
 
data_quality_log = pd.DataFrame(
    quality_entries,
    columns=quality_log_columns
)
 
display(data_quality_log)


,dataset,field,issue,affected_rows,treatment,justification,remaining_risk
0,service_requests_raw.csv,service_type,Spelling variation: electrcity,1,Mapped to Electricity,Obvious spelling error referring to the same s...,Other unknown spelling errors may still exist.
1,service_requests_raw.csv,submission_channel,Inconsistent capitalisation: Whatsapp,1,Mapped to WhatsApp,Both values refer to the same messaging platform.,Other inconsistent channel labels may exist.
2,service_requests_raw.csv,satisfaction_rating,Invalid rating,1,Flagged for investigation,Ratings must fall within the expected 1 to 5 r...,The original rating must be verified.
3,service_requests_raw.csv,resolved_at,Missing resolution dates,215,Retained missing values and flagged for invest...,Some requests may still be unresolved.,Some completed requests may have missing dates.
4,service_requests_raw.csv,resolution_hours,Missing resolution times,214,Calculated from valid timestamps where possibl...,Valid timestamps provide a reliable way to cal...,Some missing values may remain unresolved.
5,service_requests_raw.csv,created_at and resolved_at,Resolution date before creation date,1,Flagged for investigation.,A request cannot normally be resolved before i...,Original timestamps require verification.
6,service_requests_raw.csv,resolution_hours,Extreme outlier: 9999 hours (SR00039),1,Flagged for investigation; not automatically r...,The value exceeds the IQR upper limit of appro...,The value may be genuine or caused by a record...
7,resident_profiles_raw.csv,province,Abbreviation: WC,1,Mapped to Western Cape,WC is a recognised abbreviation for Western Cape.,Other province abbreviations may exist.
8,resident_profiles_raw.csv,preferred_language,Inconsistent language label: Zulu,1,Mapped to isiZulu,Both labels refer to the same language.,Other language naming inconsistencies may exist.
9,resident_profiles_raw.csv,has_smartphone,Inconsistent value: Y,1,Mapped to Yes,Y is an abbreviation for Yes in a Yes/No field.,Other unexpected responses may exist.


# Step 14 — Check referential integrity and merge data

`resident_id` links service requests to resident profiles.

A **foreign-key problem** occurs when a request refers to a resident who does not exist in the resident table.

Use `merge(..., indicator=True)` to see whether rows matched.


In [44]:
merge_check = requests_clean.merge(
    residents_clean,
    on="resident_id",
    how="left",
    indicator=True,
    suffixes=("_request", "_resident")
)

display(merge_check["_merge"].value_counts())
display(merge_check.loc[merge_check["_merge"] != "both"].head())


_merge
both          796
left_only       6
right_only      0
Name: count, dtype: int64

,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system,invalid_coordinates,resolution_before_creation,invalid_priority,invalid_status,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone,invalid_age,negative_household_income,_merge
77,SR00078,R9999,2026-06-02 17:57:00,2026-06-05 09:34:00,Roads,Call Centre,Medium,Resolved,63.62,NaN,-32.764524,30.489620,Device-007,CRM,False,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
244,SR00245,R0034,2026-02-15 22:28:00,2026-02-17 09:23:00,Sanitation,Walk-in,Medium,Resolved,34.93,4.0,-27.446932,26.628640,Device-041,Mobile API,False,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
359,SR00359,R0034,2026-03-19 18:00:00,2026-03-21 04:03:00,Water,Web Portal,High,Closed,34.06,2.0,-29.087853,21.040335,Device-004,CRM,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
393,SR00393,R0034,2026-04-01 14:38:00,2026-04-03 20:26:00,Sanitation,Call Centre,Medium,Resolved,53.80,NaN,-32.601465,24.374648,Device-001,Mobile API,False,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
664,SR00663,R0034,2026-04-09 12:38:00,NaT,Electricity,Mobile App,Medium,Open,NaN,3.0,-27.968489,31.263914,Device-074,Call Centre Export,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


### Your activity

Explain:
- how many requests did not match a resident;
- why this is a problem;
- whether you will remove, retain or flag those requests.


How many requests did not match a resident?
- Six (6) service requests did not match any resident records.
- 796 requests matched successfully with existing resident profiles.
Why is this a problem?
- These requests contain resident_id values that cannot be found in the resident profiles dataset.
- This means that the resident's information, such as name, age, province and municipality, cannot be linked to the service request.
- It may affect geographical analysis, resident demographics and the accuracy of service delivery reports.
How should these requests be treated?
- I would retain and flag the six unmatched requests for further investigation.
- I would check whether the resident IDs were entered incorrectly or whether the resident profiles are missing from the database.
- I would not delete these requests because they may contain valuable information about municipal service delivery.
- If the correct resident information becomes available, I would update the records and perform the merge again.

Then create the final merged DataFrame.


In [45]:
final_dataset = merge_check.copy()

# Add an easy-to-understand flag.
final_dataset["resident_match_found"] = final_dataset["_merge"].eq("both")

# Remove the technical merge-indicator column only after creating the flag.
final_dataset = final_dataset.drop(columns=["_merge"])

display(final_dataset.head())


,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system,invalid_coordinates,resolution_before_creation,invalid_priority,invalid_status,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone,invalid_age,negative_household_income,resident_match_found
0,SR00001,R0355,2026-05-26 15:17:00,2026-05-27 11:00:00,Water,Web Portal,High,Resolved,19.73,NaN,-33.913755,31.024099,Device-003,CRM,False,False,False,True,Resident 355,44.0,Western Cape,Garden Route District,15131.0,Sesotho,No,False,False,True
1,SR00002,R0301,2026-02-06 20:52:00,NaT,Roads,Walk-in,High,In Progress,NaN,5.0,-32.107892,29.734713,Device-070,Mobile API,False,False,False,False,Resident 301,25.0,Eastern Cape,Nelson Mandela Bay,24791.0,Afrikaans,Yes,False,False,True
2,SR00003,R0197,2026-02-05 20:44:00,2026-02-08 18:51:00,Water,Mobile App,Medium,Closed,70.12,1.0,-33.350134,21.480373,Device-044,Call Centre Export,False,False,False,False,Resident 197,59.0,KwaZulu-Natal,King Cetshwayo District,6137.0,isiZulu,No,False,False,True
3,SR00004,R0394,2026-06-07 17:25:00,NaT,Waste,Walk-in,Low,In Progress,NaN,4.0,-29.931750,23.174613,Device-056,CRM,False,False,False,False,Resident 394,73.0,Western Cape,Garden Route District,19096.0,isiXhosa,Yes,False,False,True
4,SR00005,R0349,NaT,NaT,Water,Call Centre,High,In Progress,NaN,2.0,-27.995956,19.423921,Device-049,Call Centre Export,False,False,False,False,Resident 349,82.0,Gauteng,Tshwane Metro,40226.0,isiZulu,Yes,False,False,True


# Step 15 — Write reusable functions (PA0104)

Reproducible code means:
- another person can run it again;
- the same steps are followed every time;
- there are no hidden manual changes.

A function stores a reusable set of instructions.


In [46]:
def missing_value_summary(df):
    summary = df.isna().sum().to_frame("missing_count")
    summary["missing_percentage"] = (
        summary["missing_count"] / len(df) * 100
    ).round(2)
    return summary.sort_values("missing_count", ascending=False)

display(missing_value_summary(final_dataset).head(10))


,missing_count,missing_percentage
resolved_at,215,26.81
resolution_hours,214,26.68
satisfaction_rating,212,26.43
municipality,9,1.12
age,7,0.87
resident_name,7,0.87
invalid_age,6,0.75
has_smartphone,6,0.75
negative_household_income,6,0.75
preferred_language,6,0.75


### Your activity

Create at least two more functions, for example:
- a function to identify duplicate keys;
- a function to validate numeric ranges;
- a function to standardise category text;
- a function to create an IQR outlier flag.


In [47]:
# Write your functions here.

def identify_duplicate_keys(df, column):
    duplicates = df[df.duplicated(subset=[column], keep=False)]
    return duplicates.sort_values(by=column)

# Check for duplicate resident IDs.
display(identify_duplicate_keys(residents_clean, "resident_id"))


def validate_numeric_range(df, column, minimum, maximum):
    invalid_values = df[
        ~df[column].between(minimum, maximum)
    ]
    return invalid_values

# Check whether resident ages fall between 18 and 100.
display(validate_numeric_range(residents_clean, "age", 18, 100))

,resident_id,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone,invalid_age,negative_household_income
32,R0033,Resident 33,30.0,Eastern Cape,Nelson Mandela Bay,28453,Afrikaans,No,False,False
33,R0033,Resident 34,24.0,Eastern Cape,OR Tambo District,25290,English,Yes,False,False


,resident_id,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone,invalid_age,negative_household_income
7,R0008,Resident 8,NaN,KwaZulu-Natal,King Cetshwayo District,10717,Sesotho,Yes,False,False
88,R0089,Resident 89,150.0,Eastern Cape,Nelson Mandela Bay,14412,Afrikaans,Yes,True,False


# Step 16 — Final validation

Assertions are automatic checks. The notebook stops if an important rule fails.

Only use an assertion after you have decided what the final rule should be.


In [48]:
# Example checks. Adapt them to your final cleaning decisions.

assert len(final_dataset) > 0, "The final dataset is empty."
assert "request_id" in final_dataset.columns, "request_id is missing."
assert "resident_match_found" in final_dataset.columns, "Merge flag is missing."

print("Basic validation checks passed.")


Basic validation checks passed.


### Your activity

Add assertions for:
- unique request IDs, if that is your final rule;
- no negative resolution hours;
- ratings only between 1 and 5 when present;
- required fields being present.


In [54]:
# Add final assertions here.

# Find all records with duplicate request IDs.
duplicate_requests = final_dataset[
    final_dataset["request_id"].duplicated(keep=False)
]

display(duplicate_requests.sort_values("request_id"))

print("Number of duplicate request IDs:",
      final_dataset["request_id"].duplicated().sum())

# Check whether the duplicate records are completely identical.
display(duplicate_requests)

final_dataset = final_dataset.drop_duplicates(
    subset=["request_id"],
    keep="first"
)


assert final_dataset["request_id"].is_unique, "Duplicate request IDs found."

print("Request IDs are unique.")

,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system,invalid_coordinates,resolution_before_creation,invalid_priority,invalid_status,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone,invalid_age,negative_household_income,resident_match_found


Number of duplicate request IDs: 0


,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system,invalid_coordinates,resolution_before_creation,invalid_priority,invalid_status,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone,invalid_age,negative_household_income,resident_match_found


Request IDs are unique.


# Step 17 — Save the outputs

Use `to_csv()` to save the final tables.

`index=False` prevents pandas from adding an unnecessary row-number column.


In [55]:
final_dataset.to_csv(
    OUTPUT_FOLDER / "clean_service_requests.csv",
    index=False
)

data_quality_log.to_csv(
    OUTPUT_FOLDER / "data_quality_log.csv",
    index=False
)

source_specification.to_csv(
    OUTPUT_FOLDER / "source_specification.csv",
    index=False
)

print("Files saved inside:", OUTPUT_FOLDER.resolve())


Files saved inside: C:\Users\Sanele\Downloads\PM06 python\outputs


# Step 18 — Present preliminary findings

Now that the data has been prepared, calculate simple summaries to motivate the business problem.

Examples:
- request count by service type;
- median resolution hours by service type;
- median resolution hours by submission channel;
- percentage of unmatched residents;
- percentage of requests with missing satisfaction ratings.

Do not perform complex modelling. The focus of this assignment is data preparation.


In [56]:
# Example
service_summary = (
    final_dataset.groupby("service_type", dropna=False)
    .agg(
        request_count=("request_id", "count"),
        median_resolution_hours=("resolution_hours", "median")
    )
    .sort_values("median_resolution_hours", ascending=False)
)

display(service_summary)


,request_count,median_resolution_hours
service_type,,
Electricity,168,54.330
Water,165,53.685
Sanitation,163,51.600
Roads,148,46.105
Waste,153,44.710
NaN,1,NaN


## My three preliminary observations

1.  Electricity requests have the longest median resolution time. Electricity requests have a median resolution time of 54.33 hours, followed by Water at 53.69 hours and Sanitation at 51.60 hours. This suggests that these service categories may require further investigation to identify possible delays.
2.  Electricity has the highest number of requests. There were 168 electricity requests, followed by Water with 165 and Sanitation with 163. This may indicate a greater demand for these municipal services.
3. Waste requests have the shortest median resolution time. Waste requests have a median resolution time of 44.71 hours, compared with 54.33 hours for Electricity. This suggests that waste-related requests are being resolved faster on average, although further investigation would be needed to understand why.


# Final readiness conclusion

Write 250–400 words in plain language.

Answer:
1. Is the data now ready for analysis?
2. What were the biggest quality problems?
3. Which cleaning choices may affect the results?
4. What problems remain unresolved?
5. What should CityConnect improve in its data-capture process?

_Write your conclusion here._

### Final Readiness Conclusion

The CityConnect dataset has undergone several data preparation and cleaning processes to improve its quality, consistency and reliability. The data is now better organised and provides a useful foundation for preliminary analysis. However, it is not yet fully ready for final analysis because some data quality issues still require investigation.

The biggest quality problems identified included missing values, duplicate records, inconsistent data types, invalid numerical values, unusual resolution times and referential integrity issues. For example, the satisfaction rating, resolution time and resolution date columns contained a significant number of missing values. The resident profiles also contained duplicate resident IDs, while the service centre dataset included invalid staff counts and negative monthly capacity values.

Several cleaning decisions may affect the results. These included converting columns to appropriate numerical and date formats, identifying outliers using the IQR method, flagging invalid values and removing verified duplicate records. Missing resolution times and satisfaction ratings were not automatically replaced because doing so could introduce misleading information into the analysis.

Some issues remain unresolved, particularly conflicting request IDs, unmatched resident records and missing information. These issues should be investigated against the original source data before final conclusions are drawn. The duplicate request ID assertion also needs to pass before the dataset can be considered fully validated.

Based on the preliminary findings, Electricity requests had the highest request count and the longest median resolution time of 54.33 hours. Water and Sanitation also showed relatively high median resolution times. These findings suggest that CityConnect should investigate the factors contributing to delays in these service categories.

To improve future data collection, CityConnect should introduce mandatory fields for important information, enforce unique identifiers, validate numerical values during data entry and standardise date formats and service categories. The municipality should also improve its process for linking resident profiles to service requests.

Overall, the data preparation process has improved the dataset considerably, but outstanding validation issues must be resolved before conducting reliable final analysis.

# Final checklist

- [ ] I explained the business problem.
- [ ] I loaded all three files.
- [ ] I created a source specification.
- [ ] I inspected all datasets.
- [ ] I kept the original data unchanged.
- [ ] I corrected mixed data types.
- [ ] I checked dates and categories.
- [ ] I checked missing values and duplicates.
- [ ] I created validation flags.
- [ ] I used two outlier methods.
- [ ] I checked foreign-key matches.
- [ ] I created a data-quality log.
- [ ] I used NumPy and pandas.
- [ ] I created reusable functions.
- [ ] My notebook runs from top to bottom.
- [ ] I exported all three output files.
